In [7]:
# The Marko and Maximus Hackathon Solution
# 2/20/25

# Program flow:
# Imports (software)
# Imports (data)
# Confirm data properly imported
# Fix bad data values, if applicable
# Load training and testing data
# Fitting the model
# Checking the model validation
# Testing the model
# Testing whether the model is accurate on a scatter graph
# Permutations!
# Partial dependence plots
# The actual neural network training
# Everything else
# 
# Note: Training data MUST be kept separately from testing data!

In [8]:
# %% 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import tensorflow as tf
import shap

# SciKit-Learn imports
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Ensure plots display in the notebook
%matplotlib inline

# Set numpy options for clarity
np.set_printoptions(suppress=True)


In [9]:
# %%
def import_data(uri, delim):
    """Import data from a CSV file."""
    return pd.read_csv(uri, delimiter=delim)

# Update these paths to point to your files
test_data = import_data(r"C:\RPI\Graduation Semester\Behavioral Data Science [PSYC 4961]\Code\Hackathon\mvms-marp\test_data.csv", ";")
train_data = import_data(r"C:\RPI\Graduation Semester\Behavioral Data Science [PSYC 4961]\Code\Hackathon\mvms-marp\train_data.csv", ";")

if test_data is None or train_data is None:
    print('Could not import test or training data: try again')

print('\n----Test Data Head----\n')
display(test_data.head())

print('\n----Training Data Head----\n')
display(train_data.head())



----Test Data Head----



,subject,country,rel_1,rel_2,rel_3,rel_4,rel_5,rel_6,rel_7,rel_8,...,gender,ses,education,ethnicity,denomination,gdp,gdp_scaled,sample_type,compensation,attention_check
0,4079,Germany,0.000000,0.000000,0.0,0,0.166667,0.166667,0.166667,0.00,...,woman,7.0,5,Caucasian/European,NaN,48195.579904,0.590207,mixed,raffle,1
1,1301,Canada,0.000000,0.142857,0.0,0,0.500000,0.166667,0.166667,0.00,...,woman,6.0,4,Caucasian/European,NaN,46210.547623,0.500519,online panel,monetary reward,1
2,2689,Denmark,0.833333,0.714286,1.0,1,0.666667,1.000000,0.666667,0.50,...,man,6.0,4,African,Muslim,60726.466535,1.156377,mixed,raffle,1
3,5909,Israel,0.000000,0.000000,0.0,0,0.166667,0.000000,0.000000,0.00,...,man,7.0,4,Mixed / other,NaN,41613.998082,0.292837,students,course credit,1
4,51,Australia,1.000000,1.000000,1.0,1,0.833333,1.000000,1.000000,0.75,...,man,9.0,7,Caucasian/European,Muslim,57305.299016,1.001802,online panel,monetary reward,1



----Training Data Head----



,subject,country,rel_1,rel_2,rel_3,rel_4,rel_5,rel_6,rel_7,rel_8,...,gender,ses,education,ethnicity,denomination,gdp,gdp_scaled,sample_type,compensation,attention_check
0,5362,Ireland,0.833333,0.857143,1.0,1,0.833333,0.833333,1.000000,1.00,...,woman,6.0,4,Caucasian/European,Christian (Roman Catholic),78806.431996,1.973267,online panel,monetary reward,1
1,10070,UK,0.000000,0.000000,0.0,0,0.166667,0.166667,0.166667,0.00,...,woman,5.0,3,Caucasian/European,NaN,42491.364435,0.332479,online panel,monetary reward,1
2,9515,Turkey,0.000000,0.000000,0.5,0,0.000000,0.500000,0.000000,0.00,...,man,6.0,5,Middle-Eastern/Arab,NaN,9311.366117,-1.166661,online panel,monetary reward,1
3,5960,Israel,0.500000,0.428571,0.5,0,0.166667,0.500000,0.666667,0.25,...,woman,7.0,4,Mixed / other,NaN,41613.998082,0.292837,students,course credit,1
4,6935,Lithuania,0.000000,0.000000,0.5,1,0.166667,0.666667,0.166667,0.00,...,woman,7.0,3,Caucasian/European,Christian (Roman Catholic),19089.707506,-0.724855,students,no compensation,1


In [13]:

# %%
def fix_bad_data_values(df):
    for col in df.columns:
        if df[col].dtype in [np.float64, np.int64]:
            if df[col].isna().sum() > 0:
                df.loc[:, col] = df[col].fillna(df[col].mean())
        else:
            if df[col].isna().sum() > 0:
                df.loc[:, col] = df[col].fillna(df[col].mode()[0])
    return df

train_data = fix_bad_data_values(train_data)
test_data = fix_bad_data_values(test_data)

print("Missing values in training data:\n", train_data.isna().sum())
print("Missing values in test data:\n", test_data.isna().sum())


Missing values in training data:
 subject              0
country              0
rel_1                0
rel_2                0
rel_3                0
rel_4                0
rel_5                0
rel_6                0
rel_7                0
rel_8                0
rel_9                0
cnorm_1              0
cnorm_2              0
wb_gen_1             0
wb_gen_2             0
wb_phys_1            0
wb_phys_2            0
wb_phys_3            0
wb_phys_4            0
wb_phys_5            0
wb_phys_6            0
wb_phys_7            0
wb_psych_1           0
wb_psych_2           0
wb_psych_3           0
wb_psych_4           0
wb_psych_5           0
wb_psych_6           0
wb_soc_1             0
wb_soc_2             0
wb_soc_3             0
wb_overall_mean      0
wb_phys_mean         0
wb_psych_mean        0
wb_soc_mean          0
age                  0
gender               0
ses                  0
education            0
ethnicity            0
denomination         0
gdp                  0


In [14]:
# %%
# List of religiosity variables
rel = ['rel_1','rel_2','rel_3','rel_4','rel_5','rel_6','rel_7','rel_8','rel_9']
rel_data = train_data[rel]

# Apply PCA to reduce the 9 variables to 1 principal component (keep the original PCA method)
pca = PCA(n_components=1)
rel_comp = pca.fit_transform(rel_data)

# Add the principal component as a new column in the training DataFrame
train_data['religiosity_index'] = rel_comp

print("Sample principal component (religiosity_index):", rel_comp[:5])


Sample principal component (religiosity_index): [[ 1.5614437 ]
 [-1.05046523]
 [-0.84962936]
 [-0.14181316]
 [-0.20508182]]


In [15]:
# %%
# Define target and features from training data (adjust if needed)
y = train_data['wb_overall']
X = train_data.drop(['wb_overall'], axis=1)

# Lists for feature engineering (update these lists as needed)
demographic_variables = ["age", "gdp_scaled"]
categorical_vars = ["country", "gender", "sample_type"]

# Create dummy variables for categorical features in training data
X = pd.get_dummies(X, columns=categorical_vars)

# For the test data, if wb_overall exists as target:
if 'wb_overall' in test_data.columns:
    y_test_full = test_data['wb_overall']
    X_test = test_data.drop(['wb_overall'], axis=1)
else:
    X_test = test_data.copy()

X_test = pd.get_dummies(X_test, columns=categorical_vars)

# Align training and test features (fill missing dummy columns with 0)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)


KeyError: 'wb_overall'

In [16]:
# %%
# Split into training+validation and hold-out test (20% hold-out)
X_train_val, X_test_split, y_train_val, y_test_split = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split training+validation into training and validation (20% of original becomes validation)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)  # 0.25 * 0.8 = 0.2

# Standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_split)


NameError: name 'X' is not defined

In [17]:
# %%
# Linear Regression
line_reg = LinearRegression()
line_reg.fit(X_train_scaled, y_train)
preds_train_lr = line_reg.predict(X_train_scaled)
rmse_train_lr = np.sqrt(mean_squared_error(y_train, preds_train_lr))
print("Linear Regression RMSE (Train):", rmse_train_lr)

# Bayesian Ridge Regression
bayes_ridge = BayesianRidge()
bayes_ridge.fit(X_train_scaled, y_train)
preds_train_br = bayes_ridge.predict(X_train_scaled)
rmse_train_br = np.sqrt(mean_squared_error(y_train, preds_train_br))
print("Bayesian Ridge RMSE (Train):", rmse_train_br)

# Validate models on the validation set
preds_val_lr = line_reg.predict(X_val_scaled)
rmse_val_lr = np.sqrt(mean_squared_error(y_val, preds_val_lr))
print("Linear Regression RMSE (Validation):", rmse_val_lr)

preds_val_br = bayes_ridge.predict(X_val_scaled)
rmse_val_br = np.sqrt(mean_squared_error(y_val, preds_val_br))
print("Bayesian Ridge RMSE (Validation):", rmse_val_br)


NameError: name 'X_train_scaled' is not defined

In [18]:
# %%
# Evaluate Linear Regression on test set
preds_test_lr = line_reg.predict(X_test_scaled)
rmse_test_lr = np.sqrt(mean_squared_error(y_test_split, preds_test_lr))
print("Linear Regression RMSE (Test):", rmse_test_lr)

# Evaluate Bayesian Ridge on test set
preds_test_br = bayes_ridge.predict(X_test_scaled)
rmse_test_br = np.sqrt(mean_squared_error(y_test_split, preds_test_br))
print("Bayesian Ridge RMSE (Test):", rmse_test_br)

# Scatter plot: Actual vs Predicted
plt.figure(figsize=(10, 5))
plt.scatter(y_test_split, preds_test_lr, label='Linear Regression', alpha=0.7)
plt.scatter(y_test_split, preds_test_br, label='Bayesian Ridge', alpha=0.7)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted Values (Test Set)")
plt.legend()
plt.show()


NameError: name 'X_test_scaled' is not defined

In [ ]:
# %%
# Permutation importance for Linear Regression on test set
perm_importance_lr = permutation_importance(line_reg, X_test_scaled, y_test_split, n_repeats=10, random_state=42)
print("Permutation Importance (Linear Regression):", perm_importance_lr.importances_mean)

def make_partial_dependence_plot(model, X, feature_names):
    """
    Create a partial dependence plot using SHAP.
    (For demonstration, plots the first feature.)
    """
    explainer = shap.Explainer(model.predict, X)
    shap_values = explainer(X)
    shap.plots.partial_dependence(shap_values, feature=feature_names[0], feature_names=feature_names)

# Create a partial dependence plot for the first feature (adjust as needed)
make_partial_dependence_plot(line_reg, X_test_scaled, X.columns.tolist())



In [19]:
# %%
def make_neural_network(input_shape):
    """Create and compile a simple feed-forward neural network."""
    model = tf.keras.models.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=input_shape),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Build the neural network
nn_model = make_neural_network((X_train_scaled.shape[1],))
nn_model.summary()

# Train the neural network
history = nn_model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
                       epochs=100, batch_size=32, verbose=0)

# Evaluate the neural network on the test set
nn_test_loss, nn_test_mae = nn_model.evaluate(X_test_scaled, y_test_split, verbose=0)
print("Neural Network Test Loss (MSE):", nn_test_loss)
print("Neural Network Test MAE:", nn_test_mae)


NameError: name 'X_train_scaled' is not defined

In [20]:
# %%
# Select numeric columns from training data
numeric_cols = train_data.select_dtypes(include=[np.number])
corr_matrix = np.corrcoef(numeric_cols.values, rowvar=False)
print("Correlation Matrix:\n", corr_matrix)


Correlation Matrix:
 [[ 1.          0.00789435  0.03847854 ... -0.00481701  0.04014972
   0.0180665 ]
 [ 0.00789435  1.          0.74321846 ... -0.16817047  0.14312478
   0.77363067]
 [ 0.03847854  0.74321846  1.         ... -0.1619892   0.12785332
   0.85507755]
 ...
 [-0.00481701 -0.16817047 -0.1619892  ...  1.         -0.11538448
  -0.15735991]
 [ 0.04014972  0.14312478  0.12785332 ... -0.11538448  1.
   0.14859603]
 [ 0.0180665   0.77363067  0.85507755 ... -0.15735991  0.14859603
   1.        ]]
